In [ ]:
import json
import numpy as np
import pandas as pd

def galaxy_path(name):
    return GALAXY_INPUTS[name][0]['path']

deseq_path = galaxy_path('deseq')
pseudobulk_path = galaxy_path('pseudobulk')

result_columns = ['gene', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']
de = pd.read_csv(deseq_path, sep='\t', header=None, names=result_columns)
pbulk = pd.read_csv(pseudobulk_path, sep='\t')
if list(pbulk.columns)[0] != 'gene':
    raise ValueError('Pseudobulk matrix must start with gene column')
if de.shape[1] != len(result_columns):
    raise ValueError('Unexpected DESeq2 result width')
if de['gene'].isna().any() or pbulk['gene'].isna().any():
    raise ValueError('Missing gene identifier')
if de['gene'].duplicated().any() or pbulk['gene'].duplicated().any():
    raise ValueError('Duplicate gene identifiers')
if set(de['gene'].astype(str)) != set(pbulk['gene'].astype(str)):
    raise ValueError('DESeq2 and pseudobulk gene sets differ')

lfc = pd.to_numeric(de['log2FoldChange'], errors='coerce').to_numpy(dtype=float)
pval = pd.to_numeric(de['pvalue'], errors='coerce').to_numpy(dtype=float)
if not np.isfinite(lfc).all():
    raise ValueError('Non-finite log2 fold change in DESeq2 output')
if not np.isfinite(pval).all() or (pval < 0).any() or (pval > 1).any():
    raise ValueError('Invalid p-value in DESeq2 output')

n = len(pval)
order = np.argsort(pval, kind='mergesort')
sorted_p = pval[order]
ranks = np.arange(1, n + 1, dtype=float)
sorted_q = sorted_p * n / ranks
sorted_q = np.minimum.accumulate(sorted_q[::-1])[::-1]
fdr = np.empty(n, dtype=float)
fdr[order] = np.clip(sorted_q, 0, 1)

out = pd.DataFrame({
    'gene': de['gene'].astype(str),
    'log2_fold_change': lfc,
    'p_value': pval,
    'fdr': fdr,
})
if not np.isfinite(out['fdr'].to_numpy()).all():
    raise ValueError('Non-finite adjusted p-value')
out.to_csv('outputs/differential_expression.tsv', sep='\t', index=False, lineterminator='\n')
print(json.dumps({'genes': int(n), 'columns': list(out.columns), 'min_p_value': float(pval.min()), 'max_p_value': float(pval.max()), 'min_fdr': float(fdr.min()), 'max_fdr': float(fdr.max())}))
